# 7K Challenge - Baseline

This notebook builds a **baseline reconstruction** for a **list of French power network snapshots**, and packages them into the file format expected by the **7K Challenge** submission.

## What problem are we solving?

We start from network snapshots (`.xiidm` files) with missing loads, generators and cross-border exchanges. For **each** snapshot our goal is to reconstruct plausible values for:

- **Loads** (consumption), using regional consumption figures from **eCO2mix**,
- **Generators** (production), using regional production-by-source figures from **eCO2mix**,
- **Cross-border flows** (dangling lines and HVDC generators), using net exchange figures from **ENTSO-E**,

and then to:

1. inject these reconstructed values back into the network,
2. (optionally) export the updated network to `.xiidm`,
3. aggregate the result **per substation** and export **one combined Parquet + zip** submission file covering every timestamp.

## Method used for this baseline

This is intentionally a **simple baseline**, meant as a starting point:

- **Loads**: the total regional consumption is split **uniformly** across all connected loads of that region.
- **Generators**: production is split **proportionally to each generator's installed capacity (`max_p`)**, capped so no generator exceeds its own `max_p` (surplus is redistributed to generators with remaining headroom). Nuclear is handled at the **national** level, everything else region by region.
- **Cross-border flows**: the net flow with each neighboring country is split **uniformly** across the dangling lines and HVDC generators connected to that country.

## How the batch works

You only provide a **list of timestamps** (`TIMESTAMPS` in Step 0.2). For each timestamp the notebook derives, from the shared naming convention, the matching `.xiidm` snapshot and ENTSO-E flow file. Reconstruction runs snapshot by snapshot and prints **one summary line each** all results are concatenated into a single submission.

Under the hood it is also optimized: the substation → region map is computed **once**, eCO2mix CSVs are cached (each `(region, year)` file read once for the whole run), and each snapshot's network is loaded **once** and reconstructed in memory.

## Notebook structure

| Step | What it does |
|---|---|
| 0. Setup | Imports and configuration (paths, naming convention, the `TIMESTAMPS` list, flags) |
| 1. Reading helpers | Load the XIIDM network, precompute the region map, cache eCO2mix, read ENTSO-E |
| 2. Loads | Reconstruct consumption |
| 3. Generators | Reconstruct production |
| 4. Cross-border flows | Reconstruct dangling line and HVDC generator exchanges |
| 5. Network update | Inject reconstructed values into the network |
| 6. DC load flow validation | Check physical consistency |
| 7. Submission | Aggregate per substation |
| 8. Per-snapshot pipeline | Assemble steps 1-7 into `reconstruct_snapshot()` |
| 9. Batch run | Loop over `TIMESTAMPS`, print one line each, export the combined submission |


## Data sources

This baseline combines four data sources. The XIIDM snapshots are provided by the challenge itself, the other three are public open data you can browse or re-download independently.

| Data | What it's used for | Origin |
|---|---|---|
| **Network snapshot** (`.xiidm` / `.xiidm.bz2`) | The grid topology (substations, voltage levels, loads, generators, dangling lines) we reconstruct values onto | RTE7K Dataset - [huggingface.co](https://huggingface.co/datasets/OpenSynth/D-GITT-RTE7000-2021) |
| **eCO2mix regional** (consumption + production by source, per region) | Regional consumption and production-by-filiere targets | RTE eco2mix - [rte-france.com](https://www.rte-france.com/donnees-publications/eco2mix-donnees-temps-reel/telecharger-indicateurs) |
| **ENTSO-E cross-border flows** | Net France <-> neighboring country exchange, per interconnection | ENTSO-E Transparency Platform - [transparency.entsoe.eu](https://transparency.entsoe.eu/) |
| **Postes electriques RTE** (substation geography) | Maps each substation to a French department, then to an RTE region | ODRE Open Data - [odre.opendatasoft.com](https://odre.opendatasoft.com/explore/dataset/postes-electriques-rte/) |


## Glossary

A few domain-specific terms used throughout this notebook:

- **XIIDM** — the network exchange file format used by [powsybl](https://www.powsybl.org/)/`pypowsybl` to describe a grid's topology and state (substations, voltage levels, loads, generators, lines, ...).
- **Substation / voltage level** — a substation is a physical site, each substation contains one or more voltage levels (e.g. 400 kV, 225 kV busbars), and network elements (loads, generators, ...) attach to a voltage level.
- **Dangling line** — the French-side half of a cross-border interconnection line whose other end (outside France) isn't modeled in the network; it's used to inject/withdraw the net cross-border flow.
- **HVDC generator** — a generator element (named `HVDC_FR_<country>`) used in RTE7000 to represent an HVDC interconnection that exists in the real network but is not explicitly modeled as a line, it injects or withdraws the power carried by that DC link directly onto the French bus.
- **eCO2mix** — RTE's platform reporting real-time and historical French electricity consumption and production, at national and regional granularity.
- **Filiere** — French for "energy source category" (e.g. Nucleaire, Eolien/wind, Solaire, Hydraulique, Thermique) as reported by eCO2mix.
- **RTE region** — one of France's administrative regions as used by RTE/eCO2mix for regional reporting (e.g. Ile-de-France, Bretagne, ...) — distinct from a voltage level or substation.
- **`p0` / `target_p`** — active power setpoints: `p0` for loads and dangling lines (consumption/withdrawal), `target_p` for generators (production target), in MW.
- **`max_p`** — a generator's installed/maximum capacity, used here to distribute regional production proportionally across generators.


## Step 0 - Setup

### 0.1 Imports

All third-party and local imports are grouped here, at the top of the notebook.


In [1]:
# --- Standard library ---
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path

# --- Third-party ---
import pandas as pd
import numpy as np
import pypowsybl.network as pn
import pypowsybl.loadflow as lf

# --- Local (project-specific lookup tables / mappings) ---
from constants import (
    COUNTRY_ALIASES,
    COUNTRY_FROM_INTERCONNECTION,
    DEPT_TO_REGION_RTE,
    ECO2MIX_PRODUCTION_COLS,
    ENERGY_SOURCE_TO_FILIERE,
    FILIERE_EN,
    RTE_REGION_TO_ECO2MIX_REGION,
)

'opf' extra dependencies are not installed, some features will not be available


### 0.2 Configuration

The **only thing to edit** is the `TIMESTAMPS` list: the timestamps you want to reconstruct (put your 24 snapshots here).

Each timestamp is turned into a `Snapshot` whose `.xiidm_path` and `.entsoe_flow_csv` are derived from the shared naming convention. If your files are named differently, adjust `xiidm_path_for` / `entsoe_path_for` (that is the single place that encodes the convention).


In [2]:
# ─────────────────────────────────────────────
# Static inputs (shared by every snapshot)
# ─────────────────────────────────────────────
XIIDM_DIR = Path("data/xiidm_files")                    
ENTSOE_DIR = Path("data/entsoe")                        
ECO2MIX_DIR = Path("data/eco2mix")                      
SUBSTATION_REGION_CSV = Path("data/geo/postes-electriques-rte.csv")

OUTPUT_DIR = Path("data/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_OUTPUT_FILE = OUTPUT_DIR / "submission_baseline.parquet"

SAVE_INTERMEDIATE = False
SAVE_XIIDM = True
RUN_LOADFLOW = True

# ─────────────────────────────────────────────
# Naming convention: derive the file paths from a timestamp
# ─────────────────────────────────────────────
def xiidm_path_for(timestamp: str) -> Path:
    t = pd.Timestamp(timestamp)
    return XIIDM_DIR / f"recollement-auto-{t:%Y%m%d-%H%M}-enrichi.xiidm.bz2"  

def entsoe_path_for(timestamp: str) -> Path:
    t = pd.Timestamp(timestamp)
    return ENTSOE_DIR / str(t.year) / f"entsoe_flows_{t:%Y-%m-%dT%H%M}.csv"

@dataclass(frozen=True)
class Snapshot:
    timestamp: str

    @property
    def xiidm_path(self) -> Path:
        return xiidm_path_for(self.timestamp)

    @property
    def entsoe_flow_csv(self) -> Path:
        return entsoe_path_for(self.timestamp)

    @property
    def tag(self) -> str:
        return pd.Timestamp(self.timestamp).strftime("%Y-%m-%dT%H%M")

# ─────────────────────────────────────────────
# The snapshots to reconstruct
# ─────────────────────────────────────────────
TIMESTAMPS = [
    "2021-06-19 10:00:00",
    "2022-01-09 08:00:00",
    "2022-03-13 12:00:00",
    "2022-03-23 13:00:00",
    "2022-04-01 07:00:00",
    "2022-04-04 13:00:00",
    "2022-04-22 12:00:00",
    "2022-05-01 12:00:00",
    "2022-06-07 14:00:00",
    "2022-08-14 20:00:00",
    "2022-09-24 09:00:00",
    "2022-11-03 18:00:00",
    "2023-03-19 12:00:00",
    "2023-04-18 12:00:00",
    "2021-01-08 12:00:00",
    "2021-01-18 10:00:00",
    "2022-03-23 12:00:00",
    "2021-08-08 02:00:00",
    "2022-01-17 19:00:00",
    "2023-01-09 19:00:00",
]
SNAPSHOTS = [Snapshot(ts) for ts in TIMESTAMPS]
print(f"[INFO] {len(SNAPSHOTS)} snapshot(s) configured")

[INFO] 20 snapshot(s) configured


### 0.3 Sanity check

Checking that the static inputs and every snapshot's derived files are available before we start.


In [3]:
STATIC_INPUTS = {
    "eCO2mix directory": ECO2MIX_DIR,
    "substation geography": SUBSTATION_REGION_CSV,
}

if not SNAPSHOTS:
    raise ValueError("TIMESTAMPS is empty, add at least one timestamp above")
missing = {label: p for label, p in STATIC_INPUTS.items() if not p.exists()}
for snap in SNAPSHOTS:
    if not snap.xiidm_path.exists():
        missing[f"network snapshot ({snap.tag})"] = snap.xiidm_path
    if not snap.entsoe_flow_csv.exists():
        missing[f"ENTSO-E flows ({snap.tag})"] = snap.entsoe_flow_csv
if missing:
    details = "\n".join(f"  - {label}: {path}" for label, path in missing.items())
    raise FileNotFoundError(
        f"Missing input file(s), please check the paths / naming convention above:\n{details}"
    )
print(f"[INFO] all required input paths exist ({len(SNAPSHOTS)} snapshot(s))")

[INFO] all required input paths exist (20 snapshot(s))


## Step 1 - Reading the network and auxiliary data

Before reconstructing anything, we set up helpers to load the XIIDM network and its loads / generators / dangling lines (each enriched with a `region_rte`), read regional eCO2mix values, read ENTSO-E flows, and compare totals.

Two things are made **static / cached** so they cost nothing to reuse across snapshots:

- `SUBSTATION_TO_REGION` — the substation → RTE region map, computed once from the geographic file.
- `ECO2MIX` — an `Eco2mixReader` that keeps every `(region, year)` CSV in memory after its first read.

All the detailed per-region logging is gated behind a `verbose` flag, so the batch run stays clean.


In [4]:
# ─────────────────────────────────────────────
# XIIDM network reading
# ─────────────────────────────────────────────

def load_network(network_xiidm: Path) -> pn.Network:
    """Loads a network from an XIIDM file (plain or compressed, e.g. .xiidm.bz2)."""
    return pn.load(str(network_xiidm))


def read_substation_regions(geo_csv: Path) -> pd.Series:
    """Reads the ODRE geographic file (';' separator) and returns a Series
    substation_id -> region_rte. Static (topology-independent), computed once."""
    geo = pd.read_csv(geo_csv, sep=";").drop_duplicates(subset="Code poste")
    regions = geo.set_index("Code poste")["departement"].map(DEPT_TO_REGION_RTE)

    n_missing = regions.isna().sum()
    if n_missing > 0:
        print(f"[WARN] {n_missing} substations without region in {geo_csv.name} (missing or unknown department)")
    return regions


# Precomputed once, shared by every snapshot
SUBSTATION_TO_REGION = read_substation_regions(SUBSTATION_REGION_CSV)


def add_substation_region(df: pd.DataFrame, network: pn.Network, verbose: bool = True) -> pd.DataFrame:
    """Adds substation_id (via voltage_level) and region_rte (via the precomputed
    SUBSTATION_TO_REGION lookup) to a dataframe of network elements."""
    vl_to_substation = network.get_voltage_levels()["substation_id"]

    df = df.copy()
    df["substation_id"] = df["voltage_level_id"].map(vl_to_substation)
    df["region_rte"] = df["substation_id"].map(SUBSTATION_TO_REGION)

    n_missing = df["region_rte"].isna().sum()
    if n_missing > 0 and verbose:
        print(f"[WARN] {n_missing} network elements without region")
    return df


def read_loads(network: pn.Network, verbose: bool = True) -> pd.DataFrame:
    """Extracts all loads from the network, enriched with their region."""
    df = add_substation_region(network.get_loads().reset_index(), network, verbose)
    if verbose:
        print(f"[INFO] {len(df)} loads extracted from XIIDM")
    return df


def read_generators(network: pn.Network, verbose: bool = True) -> pd.DataFrame:
    """Extracts all generators from the network, enriched with their region."""
    df = add_substation_region(network.get_generators().reset_index(), network, verbose)
    if verbose:
        print(f"[INFO] {len(df)} generators extracted from XIIDM")
    return df


def read_dangling_lines(network: pn.Network, verbose: bool = True) -> pd.DataFrame:
    """Extracts all dangling lines (cross-border connection points) from the network."""
    df = network.get_dangling_lines().reset_index()
    if verbose:
        print(f"[INFO] {len(df)} dangling lines extracted from XIIDM")
    return df


# ─────────────────────────────────────────────
# eCO2mix reading (cached across snapshots)
# ─────────────────────────────────────────────

class Eco2mixReader:
    """Reads regional eCO2mix CSVs and caches each (region, year) file in memory,
    so reconstructing many snapshots of the same year reads each file only once."""

    def __init__(self, eco2mix_dir: Path):
        self.eco2mix_dir = eco2mix_dir
        self._cache: dict[tuple[str, int], pd.DataFrame] = {}

    def _frame(self, region_rte: str, year: int) -> pd.DataFrame:
        region_file = RTE_REGION_TO_ECO2MIX_REGION[region_rte]
        key = (region_file, year)
        if key not in self._cache:
            path = self.eco2mix_dir / str(year) / f"eco2mix_{region_file}_{year}.csv"
            df = pd.read_csv(path,low_memory=False)
            df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Heures"])
            self._cache[key] = df.set_index("datetime").sort_index()
        return self._cache[key]

    def values(self, region_rte: str, timestamp: str, cols: list[str]) -> dict[str, float]:
        """eCO2mix values (in MW) for the requested columns, for a given RTE region
        at a given timestamp. Missing values ('ND', '-') default to 0."""
        ts = pd.to_datetime(timestamp)
        frame = self._frame(region_rte, ts.year)
        if ts not in frame.index:
            raise ValueError(f"No eCO2mix data for {region_rte} at {timestamp}")
        row = frame.loc[ts]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        return pd.to_numeric(row[cols], errors="coerce").fillna(0.0).to_dict()


# One shared reader for the whole notebook
ECO2MIX = Eco2mixReader(ECO2MIX_DIR)


def read_eco2mix_consumption(reader: Eco2mixReader, timestamp: str) -> pd.DataFrame:
    """Returns eCO2mix consumption for each RTE region at the given timestamp."""
    rows = [
        {"region_rte": region, "conso_mw": reader.values(region, timestamp, ["Consommation"])["Consommation"]}
        for region in RTE_REGION_TO_ECO2MIX_REGION
    ]
    return pd.DataFrame(rows)


# ─────────────────────────────────────────────
# ENTSO-E cross-border flows (per snapshot)
# ─────────────────────────────────────────────

def read_entsoe_flows(csv_path: Path, aliases: dict[str, str]) -> dict[str, float]:
    """Returns the net France -> country flow (in MW) per neighboring country."""
    df = pd.read_csv(csv_path)
    df["country"] = df["country"].map(lambda c: aliases.get(str(c).strip(), str(c).strip()))
    df["net_fr_to_country"] = pd.to_numeric(df["net_fr_to_country"], errors="coerce").fillna(0.0)
    return df.groupby("country")["net_fr_to_country"].sum().to_dict()


# ─────────────────────────────────────────────
# Reconstruction vs reference comparison
# ─────────────────────────────────────────────

def compare_global(reconstruction_total: float, reference_total: float, label: str = "") -> None:
    """Prints a quick side-by-side comparison between a reconstructed total and its
    reference value, along with the absolute and relative gap."""
    print(f"\n[COMPARE]{' ' + label if label else ''}")
    print(f"reference : {reference_total:.2f} MW")
    print(f"reconstruction : {reconstruction_total:.2f} MW")
    if reference_total > 0:
        gap_pct = (reconstruction_total - reference_total) / reference_total * 100
        print(f"gap : {reconstruction_total - reference_total:+.2f} MW ({gap_pct:+.2f} %)")

[WARN] 141 substations without region in postes-electriques-rte.csv (missing or unknown department)


## Step 2 - Reconstructing loads (consumption)

For each RTE region, we take the total regional consumption reported by eCO2mix and split it **uniformly** across all connected loads of that region (`conso_mw / nb_loads`). The result is later exported to `load_baseline.csv` (columns: `id`, `p0`).


In [5]:
def apply_uniform_load_disaggregation(df_load: pd.DataFrame, df_conso: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """Distributes each region's total consumption uniformly across its connected loads."""
    df = df_load.copy()
    df["p0"] = 0.0

    for _, row in df_conso.iterrows():
        region, conso_mw = row["region_rte"], row["conso_mw"]

        mask = (df["region_rte"] == region) & df["connected"]
        nb_loads = mask.sum()

        if nb_loads == 0:
            if verbose:
                print(f"[WARN] {region}: no connected load")
            continue

        df.loc[mask, "p0"] = conso_mw / nb_loads
        if verbose:
            print(f"[INFO] {region}: {conso_mw:.2f} MW distributed over {nb_loads} loads")

    return df

## Step 3 - Reconstructing generators (production)

Production is reconstructed source by source:

- **Nuclear** is handled at the **national** level (single French fleet): the national nuclear output from eCO2mix is distributed across all connected nuclear generators, proportionally to their `max_p`.
- **Every other source** (thermal, wind, solar, hydro, ...) is handled **region by region**: for each RTE region and each eCO2mix "filiere", the corresponding production is distributed across the matching connected generators, proportionally to their `max_p`.

In both cases, the distribution is capped so no generator ever exceeds its own `max_p`; any surplus that can't fit is redistributed among generators that still have headroom.

HVDC generators (name starting with `HVDC`) are excluded here (`split_standard_generators`) and handled in Step 4. Any production reported by eCO2mix that cannot be placed regionally is redistributed onto the generators that have no geographic region. The result is exported to `gen_baseline.csv` (columns: `id`, `target_p`).


In [6]:
def distribute_proportional_to_max_p(max_p: pd.Series, total_prod: float) -> tuple[pd.Series, float]:
    """Distributes total_prod proportionally to max_p, never exceeding max_p:
    the surplus from saturated generators is redistributed over the others.
    Returns the allocation and the amount that could not be allocated."""
    target_p = (total_prod * max_p / max_p.sum()).clip(upper=max_p)

    deficit = total_prod - target_p.sum()
    headroom = max_p - target_p

    if deficit > 0 and headroom.sum() > 0:
        target_p = (target_p + deficit * headroom / headroom.sum()).clip(upper=max_p)

    return target_p, total_prod - target_p.sum()


def national_nuclear_production(reader: Eco2mixReader, timestamp: str) -> float:
    """Sums nuclear production across all RTE regions to get the national total."""
    return sum(
        reader.values(region, timestamp, ["Nucléaire"])["Nucléaire"]
        for region in RTE_REGION_TO_ECO2MIX_REGION
    )


def split_standard_generators(df_gen: pd.DataFrame) -> pd.DataFrame:
    """Drops HVDC generators (name starts with 'HVDC')."""
    is_hvdc = df_gen.get("name", pd.Series("", index=df_gen.index)).str.startswith("HVDC", na=False)
    return df_gen[~is_hvdc]


def reconstruct_generators(df_gen: pd.DataFrame, reader: Eco2mixReader, timestamp: str, verbose: bool = True) -> pd.DataFrame:
    """Reconstructs target_p for every generator: nuclear nationally, everything
    else region by region and filiere by filiere. Production that eCO2mix reports
    but that could not be placed regionally is redistributed onto the generators
    without a geographic region."""
    df = df_gen.copy()
    df["target_p"] = 0.0
    df["filiere"] = df["energy_source"].map(ENERGY_SOURCE_TO_FILIERE)

    # Nuclear: national distribution
    mask_nuc = (df["energy_source"] == "NUCLEAR") & df["connected"] & (df["max_p"] > 0)
    total_prod_nuc = national_nuclear_production(reader, timestamp)

    if mask_nuc.sum() > 0:
        allocated, _ = distribute_proportional_to_max_p(df.loc[mask_nuc, "max_p"], total_prod_nuc)
        df.loc[mask_nuc, "target_p"] = allocated
        if verbose:
            print(f"[INFO] Nuclear (national): {total_prod_nuc:.2f} MW distributed over {mask_nuc.sum()} generators")

    # Other energy types: region-by-region distribution
    for region in df["region_rte"].dropna().unique():
        eco_values = reader.values(region, timestamp, ECO2MIX_PRODUCTION_COLS)

        for filiere, total_prod in eco_values.items():
            if filiere == "Nucléaire":
                continue

            mask = (
                (df["region_rte"] == region)
                & (df["filiere"] == filiere)
                & df["connected"]
                & (df["max_p"] > 0)
            )

            if mask.sum() == 0:
                if total_prod > 0 and verbose:
                    print(f"[WARN] {region} / {FILIERE_EN.get(filiere, filiere)}: {total_prod:.2f} MW but no matching generator")
                continue

            allocated, lost = distribute_proportional_to_max_p(df.loc[mask, "max_p"], total_prod)
            df.loc[mask, "target_p"] = allocated
            if verbose:
                print(f"[INFO] {region} / {FILIERE_EN.get(filiere, filiere)}: {total_prod:.2f} MW distributed over {mask.sum()} generators")

    # Redistribute unallocated surplus onto generators without a geographic region
    surplus_by_filiere = {}
    for filiere in sorted(f for f in df["filiere"].dropna().unique() if f != "Nucléaire"):
        mask_no_region = (
            df["region_rte"].isna()
            & (df["filiere"] == filiere)
            & df["connected"]
            & (df["max_p"] > 0)
        )
        if mask_no_region.sum() == 0:
            continue

        eco2mix_filiere = sum(
            reader.values(region, timestamp, ECO2MIX_PRODUCTION_COLS).get(filiere, 0.0)
            for region in df["region_rte"].dropna().unique()
        )
        surplus = eco2mix_filiere - df.loc[df["filiere"] == filiere, "target_p"].sum()
        if surplus <= 0.5:
            continue

        allocated, _ = distribute_proportional_to_max_p(df.loc[mask_no_region, "max_p"], surplus)
        df.loc[mask_no_region, "target_p"] = allocated
        surplus_by_filiere[filiere] = (surplus, mask_no_region.sum())

    if surplus_by_filiere and verbose:
        parts = ", ".join(
            f"{FILIERE_EN.get(fil, fil)}: {mw:.1f} MW over {n} gen"
            for fil, (mw, n) in surplus_by_filiere.items()
        )
        print(f"[INFO] No-region redistribution: {parts}")

    return df.drop(columns="filiere")


def print_comparison(df_reco: pd.DataFrame, reader: Eco2mixReader, timestamp: str) -> None:
    """Compares the reconstructed production (overall and nuclear-only) to the
    eCO2mix reference totals."""
    eco2mix_nuclear = national_nuclear_production(reader, timestamp)

    eco2mix_total = eco2mix_nuclear
    for region in df_reco["region_rte"].dropna().unique():
        eco_values = reader.values(region, timestamp, ECO2MIX_PRODUCTION_COLS)
        eco2mix_total += sum(v for filiere, v in eco_values.items() if filiere != "Nucléaire")

    nuclear_total = df_reco.loc[df_reco["energy_source"] == "NUCLEAR", "target_p"].sum()

    compare_global(df_reco["target_p"].sum(), eco2mix_total, label="total production")
    compare_global(nuclear_total, eco2mix_nuclear, label="nuclear")

## Step 4 - Reconstructing cross-border flows (dangling lines + HVDC generators)

Cross-border exchanges are carried by two kinds of elements: dangling lines (AC interconnections) and HVDC generators (DC links, name `HVDC_FR_<country>`). We:

1. detect which neighboring country each element belongs to - from its name prefix (`COUNTRY_FROM_INTERCONNECTION`) for dangling lines, from the `HVDC_FR_<country>` convention for HVDC generators - with country aliasing (`COUNTRY_ALIASES`),
2. read the net France -> country flow from the ENTSO-E export,
3. split that net flow **uniformly** across the connected dangling lines **and** HVDC generators of that country.

The sign convention differs between the two: for dangling lines `p0 > 0` means export France -> country, whereas for HVDC generators `target_p > 0` means an injection into the French grid (import). The results are exported to `dangling_line_baseline.csv` (columns: `id`, `p0`, `q0`) and `hvdc_gen_baseline.csv` (columns: `id`, `target_p`).


In [7]:
def detect_country(line_name: str) -> str | None:
    """Infers the neighboring country of a dangling line from its name prefix."""
    for prefix, country in COUNTRY_FROM_INTERCONNECTION.items():
        if str(line_name).startswith(prefix):
            return country
    return None


def read_hvdc_generators(network: pn.Network) -> pd.DataFrame:
    """Extracts the fictitious HVDC generators (name starting with 'HVDC') and maps
    each one to its neighboring country via the HVDC_FR_<country> naming convention."""
    gen = network.get_generators().reset_index()
    hvdc = gen[gen["name"].str.startswith("HVDC", na=False)].copy()
    hvdc["country"] = (
        hvdc["name"]
        .str.replace("HVDC_FR_", "", regex=False)
        .map(COUNTRY_ALIASES)
    )
    return hvdc[["id", "name", "country", "connected"]]


def apply_uniform_flow_disaggregation(dl: pd.DataFrame, hvdc_gen: pd.DataFrame, flows_by_country: dict[str, float], verbose: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Distributes each country's net exchange flow uniformly across its connected
    dangling lines and HVDC generators. Dangling lines carry p0 > 0 for exports,
    HVDC generators carry target_p > 0 for imports (opposite sign)."""
    dl = dl.copy()
    dl["p0"] = 0.0
    dl["q0"] = 0.0
    dl["country"] = dl["name"].apply(detect_country)

    hvdc_gen = hvdc_gen.copy()
    hvdc_gen["target_p"] = 0.0

    if verbose:
        print(f"[INFO] dangling lines  — no country: {dl['country'].isna().sum()} | disconnected: {(~dl['connected']).sum()}")
        print(f"[INFO] HVDC generators — no country: {hvdc_gen['country'].isna().sum()} | disconnected: {(~hvdc_gen['connected']).sum()}")

    valid_dl  = dl["connected"]       & dl["country"].isin(flows_by_country)
    valid_gen = hvdc_gen["connected"] & hvdc_gen["country"].isin(flows_by_country)

    for country in set(dl.loc[valid_dl, "country"]) | set(hvdc_gen.loc[valid_gen, "country"]):
        flux = flows_by_country[country]

        dl_idx  = dl.index[valid_dl  & (dl["country"]       == country)]
        gen_idx = hvdc_gen.index[valid_gen & (hvdc_gen["country"] == country)]
        n_total = len(dl_idx) + len(gen_idx)

        dl.loc[dl_idx, "p0"]              =  flux / n_total
        hvdc_gen.loc[gen_idx, "target_p"] = -flux / n_total

        if verbose:
            print(
                f"[INFO] FR <-> {country}: flow={flux:.2f} MW distributed over "
                f"{len(dl_idx)} dangling lines + {len(gen_idx)} HVDC generators"
            )

    return dl, hvdc_gen

## Step 5 - Injecting the reconstruction into the network

We write the reconstructed values back into the network (in memory, no CSV round-trip):

- **batteries** are set to zero (out of scope for this baseline),
- **loads** get their reconstructed `p0` (reactive power `q0` set to 0),
- **generators** get their reconstructed `target_p` (standard + HVDC generators concatenated; `target_q` set to 0, voltage setpoint `target_v` set to the nominal voltage of their voltage level),
- **dangling lines** get their reconstructed `p0`.

Each update function takes a dataframe indexed by element `id` and returns how many elements it touched.


In [8]:
def update_batteries_to_zero(network: pn.Network) -> int:
    """Sets all batteries' target_p / target_q to zero (out of scope for this baseline)."""
    batteries = network.get_batteries()
    if batteries.empty:
        return 0

    update = pd.DataFrame({"target_p": 0.0, "target_q": 0.0}, index=batteries.index)
    network.update_batteries(update)
    return len(update)


def update_loads(network: pn.Network, load_rec: pd.DataFrame) -> int:
    """Writes reconstructed p0 back into the network's loads (indexed by id)."""
    idx = network.get_loads().index.intersection(load_rec.index)

    update = pd.DataFrame({"p0": load_rec.loc[idx, "p0"], "q0": 0.0}, index=idx)
    network.update_loads(update)
    return len(idx)


def update_generators(network: pn.Network, gen_rec: pd.DataFrame) -> int:
    """Writes reconstructed target_p back into the network's generators, setting
    the voltage setpoint to the nominal voltage of each generator's voltage level."""
    gen_net = network.get_generators()
    idx = gen_net.index.intersection(gen_rec.index)

    nominal_v = network.get_voltage_levels()["nominal_v"]
    gen_target_v = gen_net.loc[idx, "voltage_level_id"].map(nominal_v)

    update = pd.DataFrame({
        "target_p": gen_rec.loc[idx, "target_p"],
        "target_q": 0.0,
        "target_v": gen_target_v,
    }, index=idx)

    network.update_generators(update)
    return len(idx)


def update_dangling_lines(network: pn.Network, dl_rec: pd.DataFrame) -> int:
    """Writes reconstructed p0 / q0 back into the network's dangling lines."""
    idx = network.get_dangling_lines().index.intersection(dl_rec.index)

    update = pd.DataFrame({"p0": dl_rec.loc[idx, "p0"], "q0": dl_rec.loc[idx, "q0"]}, index=idx)
    network.update_dangling_lines(update)
    return len(idx)

## Step 6 - DC load flow validation

Running a DC load flow on the reconstructed network is a quick sanity check: if the baseline values are globally consistent (balanced active power, no topological issue), the solver converges. Convergence does not mean all line ratings are respected, but a non-convergence would indicate a fundamental reconstruction error.

We also report the number of overloaded lines (current > permanent rating, derived from `I = |P₁|×10³ / (√3×V_kV)`). This is wrapped in `validate_dc_loadflow`, which returns a small summary dict per snapshot.


In [9]:
def validate_dc_loadflow(network: pn.Network, verbose: bool = True) -> dict:
    """Runs a DC load flow on the reconstructed network and counts overloaded
    lines (current > permanent rating). Returns a small summary dict."""
    params = lf.Parameters(
        distributed_slack=True,
        dc_use_transformer_ratio=True,
        balance_type=lf.BalanceType.PROPORTIONAL_TO_GENERATION_P_MAX,
    )
    results = lf.run_dc(network, parameters=params)
    status = str(results[0].status)
    converged = "CONVERGED" in status.upper()

    # --- Line overload check ---
    lines = network.get_lines()
    nominal_v = network.get_voltage_levels()["nominal_v"]
    v_kv = lines["voltage_level1_id"].map(nominal_v)
    p1_mw = pd.to_numeric(lines["p1"], errors="coerce").abs()
    i1_a = np.where(v_kv > 0, p1_mw * 1e3 / (np.sqrt(3) * v_kv), np.nan)

    op = network.get_operational_limits().reset_index()
    patl = (
        op[
            (op["type"].astype(str).str.upper() == "CURRENT")
            & (op["acceptable_duration"].astype(str) == "-1")
            & (op["element_type"].astype(str).str.upper() == "LINE")
        ]
        .drop_duplicates("element_id")
        .set_index("element_id")["value"]
    )

    df_lines = pd.DataFrame({"i1_a": i1_a, "patl_a": lines.index.map(patl)}, index=lines.index)
    df_lines["loading_pct"] = df_lines["i1_a"] / df_lines["patl_a"] * 100
    n_overloads = int((df_lines["loading_pct"] > 100).sum())
    n_rated = int(df_lines["patl_a"].notna().sum())

    if verbose:
        print(f"DC load flow status : {status}")
        print(f"Lines with permanent rating : {n_rated} / {len(lines)}")
        pct = n_overloads / n_rated * 100 if n_rated else 0.0
        print(f"Overloaded lines            : {n_overloads} ({pct:.1f}% of rated lines)")

    return {"status": status, "converged": converged, "n_rated": n_rated, "n_overloads": n_overloads}

## Step 7 - Exporting the challenge submission file

The RTE7K Challenge expects a **per-substation net power** submission. For each substation we aggregate:

```
net_p_mw = sum(generators.target_p) + sum(batteries.target_p) - sum(loads.p0) - sum(dangling_lines.p0)
```

with the challenge convention: **positive = net generation, negative = net consumption**.

`build_submission` takes the reconstructed **network object** (no reload) and returns one dataframe with columns `datetime`, `substation_id`, `net_p_mw` for a single timestamp. The batch step (9) concatenates these across all snapshots and writes a single Parquet + zip.


In [10]:
def collect_contributions(network: pn.Network) -> pd.DataFrame:
    """Returns each network element's contribution to its substation:
    positive for injections (generators, batteries), negative for
    withdrawals (loads, dangling lines). Only connected elements are included."""
    vl_to_substation = network.get_voltage_levels()["substation_id"]

    tables = [
        (network.get_generators(), "target_p", +1),
        (network.get_batteries(), "target_p", +1),
        (network.get_loads(), "p0", -1),
        (network.get_dangling_lines(), "p0", -1),
    ]

    records = []
    for table, col, sign in tables:
        if table.empty:
            continue
        table = table[table["connected"] == True]
        if table.empty:
            continue
        records.append(pd.DataFrame({
            "substation_id": table["voltage_level_id"].map(vl_to_substation),
            "p_mw": sign * table[col].fillna(0.0),
        }))

    return pd.concat(records, ignore_index=True)


def build_submission(network: pn.Network, timestamp: str) -> pd.DataFrame:
    """Builds the per-substation net_p_mw submission dataframe for one timestamp."""
    net = (
        collect_contributions(network)
        .groupby("substation_id", as_index=False)["p_mw"]
        .sum()
        .rename(columns={"p_mw": "net_p_mw"})
    )

    net.insert(0, "datetime", pd.Timestamp(timestamp, tz="Europe/Paris"))
    net["substation_id"] = net["substation_id"].astype(str)
    net["net_p_mw"] = net["net_p_mw"].astype("float64")

    assert not net["substation_id"].duplicated().any(), "duplicated substation_id"
    assert net["net_p_mw"].notna().all(), "net_p_mw contains NaN"
    assert (net["net_p_mw"].abs() <= 100_000).all(), "net_p_mw out of [-100000, 100000]"

    return net[["datetime", "substation_id", "net_p_mw"]]

## Step 8 - Per-snapshot pipeline

`reconstruct_snapshot` assembles steps 1-7 for a single snapshot. The network is loaded **once** and everything (reconstruction, injection, load flow, submission) runs on that same in-memory object.

It returns the submission dataframe and a `report` dict with the four sanity totals we care about (consumption, production, interconnection net, balance), the load flow status and overload count. By default `verbose=False` so it stays silent; the batch step prints one clean line from the report. To debug a single snapshot in detail, call `reconstruct_snapshot(SNAPSHOTS[0], verbose=True)`.


In [11]:
def reconstruct_snapshot(snap: Snapshot, verbose: bool = False) -> tuple[pd.DataFrame, dict]:
    """Full baseline reconstruction for one snapshot, in memory (the network is
    loaded exactly once). Returns the submission dataframe and a summary report."""
    t0 = time.perf_counter()
    network = load_network(snap.xiidm_path)

    # --- Loads ---
    df_load = read_loads(network, verbose)
    df_conso = read_eco2mix_consumption(ECO2MIX, snap.timestamp)
    load_rec = apply_uniform_load_disaggregation(df_load, df_conso, verbose)
    if verbose:
        compare_global(load_rec["p0"].sum(), df_conso["conso_mw"].sum(), label="consumption")

    # --- Generators (excluding HVDC) ---
    df_gen = split_standard_generators(read_generators(network, verbose))
    gen_rec = reconstruct_generators(df_gen, ECO2MIX, snap.timestamp, verbose)
    if verbose:
        print_comparison(gen_rec, ECO2MIX, snap.timestamp)

    # --- Cross-border flows (dangling lines + HVDC generators) ---
    dl = read_dangling_lines(network, verbose)
    hvdc_gen = read_hvdc_generators(network)
    flows = read_entsoe_flows(snap.entsoe_flow_csv, COUNTRY_ALIASES)
    dl_rec, hvdc_rec = apply_uniform_flow_disaggregation(dl, hvdc_gen, flows, verbose)

    # --- Assemble reconstruction frames (indexed by id) ---
    load_df = load_rec[["id", "p0"]].set_index("id")
    gen_df = pd.concat([gen_rec[["id", "target_p"]], hvdc_rec[["id", "target_p"]]]).set_index("id")
    dl_df = dl_rec[["id", "p0", "q0"]].set_index("id")

    # --- Inject into the network ---
    n_bat = update_batteries_to_zero(network)
    n_load = update_loads(network, load_df)
    n_gen = update_generators(network, gen_df)
    n_dl = update_dangling_lines(network, dl_df)
    if verbose:
        print(f"[INFO] injected — batteries:{n_bat} loads:{n_load} generators:{n_gen} dangling:{n_dl}")

    # --- Optional per-snapshot artifacts (same schema as the single-snapshot version) ---
    if SAVE_INTERMEDIATE or SAVE_XIIDM:
        snap_dir = OUTPUT_DIR / "snapshots" / snap.tag
        snap_dir.mkdir(parents=True, exist_ok=True)

    if SAVE_INTERMEDIATE:
        load_rec[["id", "p0"]].to_csv(snap_dir / "load_baseline.csv", index=False)
        gen_rec[["id", "target_p"]].to_csv(snap_dir / "gen_baseline.csv", index=False)
        hvdc_rec[["id", "target_p"]].to_csv(snap_dir / "hvdc_gen_baseline.csv", index=False)
        dl_rec[["id", "p0", "q0"]].to_csv(snap_dir / "dangling_line_baseline.csv", index=False)

    if SAVE_XIIDM:
        network.save(str(snap_dir / "baseline_reconstructed.xiidm"), format="XIIDM")

    # --- Validation ---
    report = {"timestamp": snap.timestamp}
    if RUN_LOADFLOW:
        report.update(validate_dc_loadflow(network, verbose))

    # --- Submission + the four sanity totals ---
    submission = build_submission(network, snap.timestamp)
    report["conso_mw"] = float(load_rec["p0"].sum())
    report["prod_mw"] = float(gen_rec["target_p"].sum())
    
    # net cross-border exchange, France point of view: > 0 export, < 0 import
    report["interco_mw"] = float(dl_rec["p0"].sum() - hvdc_rec["target_p"].sum())
    report["n_substations"] = len(submission)
    report["balance_mw"] = float(submission["net_p_mw"].sum())
    
    return submission, report


def format_snapshot_line(i: int, n: int, snap: Snapshot, report: dict) -> str:
    """One clean summary line per snapshot for the batch run."""
    if "converged" in report:
        lf_txt = "converged" if report["converged"] else "NOT converged"
        overloads = f"{report['n_overloads']} line overloads"
    else:
        lf_txt, overloads = "load flow skipped", "-"
    return (
        f"[{i:2d}/{n}] {pd.Timestamp(snap.timestamp):%Y-%m-%d %H:%M}  OK  "
        f"| load {report['conso_mw']:>7.0f} MW  "
        f"| gen {report['prod_mw']:>7.0f} MW  "
        f"| exchange {report['interco_mw']:>+7.0f} MW  "
        f"| LF {lf_txt}  "
        f"| {overloads}"
    )

## Step 9 - Batch run over all snapshots

We loop over every timestamp, reconstruct, and print **one line per snapshot**: reconstructed consumption, production, net interconnection, load flow convergence and overloaded-line count. All submissions are concatenated into a **single** Parquet covering every timestamp, then zipped.

A `try/except` around each snapshot means one bad file won't abort the whole campaign; failures are listed in the final `report_df` instead.


In [12]:
t_start = time.perf_counter()
submissions = []
reports = []

for i, snap in enumerate(SNAPSHOTS, start=1):
    try:
        submission, report = reconstruct_snapshot(snap, verbose=False)
        submissions.append(submission)
        reports.append(report)
        print(format_snapshot_line(i, len(SNAPSHOTS), snap, report))
    except Exception as exc:
        print(f"[{i:2d}/{len(SNAPSHOTS)}] {pd.Timestamp(snap.timestamp):%Y-%m-%d %H:%M}  FAILED: {exc}")
        reports.append({"timestamp": snap.timestamp, "status": f"ERROR: {exc}"})

if not submissions:
    raise RuntimeError("No snapshot was reconstructed successfully")

# One combined submission covering every timestamp
df = pd.concat(submissions, ignore_index=True)
df.to_parquet(SUBMISSION_OUTPUT_FILE, index=False)

zip_path = SUBMISSION_OUTPUT_FILE.with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(SUBMISSION_OUTPUT_FILE, SUBMISSION_OUTPUT_FILE.name)

print(f"\n[OK] {SUBMISSION_OUTPUT_FILE} — {df['datetime'].nunique()} timestamp(s), {len(df)} rows")
print(f"[OK] {zip_path}")
print(f"[INFO] total time: {time.perf_counter() - t_start:.1f}s for {len(submissions)}/{len(SNAPSHOTS)} snapshot(s)")

# Per-snapshot report table
report_df = pd.DataFrame(reports)
report_df

[ 1/20] 2021-06-19 10:00  OK  | load   43322 MW  | gen   48822 MW  | exchange   +6026 MW  | LF converged  | 38 line overloads
[ 2/20] 2022-01-09 08:00  OK  | load   58475 MW  | gen   63969 MW  | exchange   +2857 MW  | LF converged  | 20 line overloads
[ 3/20] 2022-03-13 12:00  OK  | load   59483 MW  | gen   54166 MW  | exchange   -6892 MW  | LF converged  | 15 line overloads
[ 4/20] 2022-03-23 13:00  OK  | load   59274 MW  | gen   53428 MW  | exchange   -4905 MW  | LF converged  | 14 line overloads
[ 5/20] 2022-04-01 07:00  OK  | load   64465 MW  | gen   54990 MW  | exchange   -8916 MW  | LF converged  | 22 line overloads
[ 6/20] 2022-04-04 13:00  OK  | load   69512 MW  | gen   64305 MW  | exchange   -3559 MW  | LF converged  | 25 line overloads
[ 7/20] 2022-04-22 12:00  OK  | load   53279 MW  | gen   52428 MW  | exchange    +232 MW  | LF converged  | 22 line overloads
[ 8/20] 2022-05-01 12:00  OK  | load   42907 MW  | gen   43408 MW  | exchange   -1549 MW  | LF converged  | 16 line ov

,timestamp,status,converged,n_rated,n_overloads,conso_mw,prod_mw,interco_mw,n_substations,balance_mw
0,2021-06-19 10:00:00,ComponentStatus.CONVERGED,True,7773,38,43322.0,48822.000000,6025.6860,3584,-525.686000
1,2022-01-09 08:00:00,ComponentStatus.CONVERGED,True,7762,20,58475.0,63968.677999,2857.4290,3604,2636.248999
2,2022-03-13 12:00:00,ComponentStatus.CONVERGED,True,7770,15,59483.0,54166.000000,-6892.4265,3618,1575.426500
3,2022-03-23 13:00:00,ComponentStatus.CONVERGED,True,7772,14,59274.0,53427.830002,-4905.4375,3618,-940.732498
4,2022-04-01 07:00:00,ComponentStatus.CONVERGED,True,7776,22,64465.0,54990.000000,-8915.7280,3627,-559.272000
5,2022-04-04 13:00:00,ComponentStatus.CONVERGED,True,7776,25,69512.0,64305.460001,-3558.8475,3630,-1647.692499
6,2022-04-22 12:00:00,ComponentStatus.CONVERGED,True,7779,22,53279.0,52427.830002,231.9740,3622,-1083.143998
7,2022-05-01 12:00:00,ComponentStatus.CONVERGED,True,7786,16,42907.0,43408.140004,-1549.4735,3609,2050.613504
8,2022-06-07 14:00:00,ComponentStatus.CONVERGED,True,7789,32,51080.0,52258.000000,2473.5830,3595,-1295.583000
9,2022-08-14 20:00:00,ComponentStatus.CONVERGED,True,7804,21,39261.0,36694.000000,-2111.4740,3592,-455.526000


## Wrap-up

You now have, for the whole list of timestamps:

- `data/output/snapshots/<timestamp>/load_baseline.csv`, `gen_baseline.csv`, `hvdc_gen_baseline.csv`, `dangling_line_baseline.csv` : the intermediate per-element reconstructions (if `SAVE_INTERMEDIATE`),
- `data/output/snapshots/<timestamp>/baseline_reconstructed.xiidm` : each network updated with its values (if `SAVE_XIIDM`),
- `data/output/submission_baseline.parquet(.zip)` : the **single** submission file (all timestamps) to submit to the 7K Challenge,
- `report_df` : a per-snapshot summary (consumption, production, interconnection, balance, load flow status, overloads).

To reconstruct more snapshots, just add their timestamps to `TIMESTAMPS` in Step 0.2. To go faster, set `SAVE_XIIDM = False` and/or `RUN_LOADFLOW = False`.
